In [1]:
import tensorflow as tf
import numpy as np
import os

# Paths #
DATA_ROOT = r"E:\Research\Vortex-small\Vortex-small"
TARGET_FOLDER = "IR_drop"

SINGLE_CHANNEL_FOLDERS = [
    "power_all",
    "power_i",
    "power_s",
    "power_sca"
]

MULTI_CHANNEL_FOLDER = "power_t"  # 20 time slices

TARGET_SIZE = (320, 320)
BATCH_SIZE = 16 

In [2]:
file_list = sorted(os.listdir(os.path.join(DATA_ROOT, TARGET_FOLDER)))
print("Number of samples:", len(file_list))
print("Example filename:", file_list[0])

Number of samples: 96
Example filename: Vortex-small_freq_200_mp_1_fpu_50_fpa_1.0_p_1_fi_ap.npz


In [3]:
def load_npy(path):
    def _load(path):
        arr_file = np.load(path.decode("utf-8"))
        
        # If npz, extract first array
        if isinstance(arr_file, np.lib.npyio.NpzFile):
            arr = list(arr_file.values())[0]
        else:
            arr = arr_file

        arr = arr.astype(np.float32)

        # If single-channel 2D, add channel dim
        if arr.ndim == 2:
            arr = arr[..., np.newaxis]

        return arr

    tensor = tf.numpy_function(_load, [path], tf.float32)
    tensor.set_shape([None, None, None])  # H, W, C unknown
    tensor = tf.image.resize(tensor, TARGET_SIZE)
    return tensor

In [4]:
def load_sample(filename):
    # Load target y #
    y_path = tf.strings.join([DATA_ROOT, TARGET_FOLDER, filename], separator=os.sep)
    y = load_npy(y_path)  # shape: (300, 300, 1)

    power_maps = []

    # Load 4 single-channel power maps #
    for folder in SINGLE_CHANNEL_FOLDERS:
        x_path = tf.strings.join([DATA_ROOT, folder, filename], separator=os.sep)
        x = load_npy(x_path)  # shape: (300, 300, 1)
        power_maps.append(x)

    # Load multi-channel power_t #
    power_t_path = tf.strings.join([DATA_ROOT, MULTI_CHANNEL_FOLDER, filename], separator=os.sep)
    
    def load_power_t(path):
        arr_file = np.load(path.decode("utf-8"))
        if isinstance(arr_file, np.lib.npyio.NpzFile):
            arr = list(arr_file.values())[0]
        else:
            arr = arr_file

        arr = arr.astype(np.float32)


        # MAIN CHANGE: transpose (20, H, W) → (H, W, 20) #
        if arr.ndim == 3 and arr.shape[0] == 20:
            arr = np.transpose(arr, (1, 2, 0))

        return arr

    power_t = tf.numpy_function(load_power_t, [power_t_path], tf.float32)
    power_t.set_shape([None, None, 20])
    power_t = tf.image.resize(power_t, TARGET_SIZE)

    power_maps.append(power_t)

    # Concatenate all channels → (300, 300, 24) #
    X = tf.concat(power_maps, axis=-1)

    return X, y

In [5]:
dataset = tf.data.Dataset.from_tensor_slices(file_list)
dataset = dataset.map(load_sample, num_parallel_calls=1)
dataset = dataset.shuffle(10)
dataset = dataset.batch(BATCH_SIZE)
dataset = dataset.prefetch(1)

In [6]:
for X, y in dataset:
    print("X shape:", X.shape)
    print("y shape:", y.shape)

X shape: (16, 320, 320, 24)
y shape: (16, 320, 320, 1)
X shape: (16, 320, 320, 24)
y shape: (16, 320, 320, 1)
X shape: (16, 320, 320, 24)
y shape: (16, 320, 320, 1)
X shape: (16, 320, 320, 24)
y shape: (16, 320, 320, 1)
X shape: (16, 320, 320, 24)
y shape: (16, 320, 320, 1)
X shape: (16, 320, 320, 24)
y shape: (16, 320, 320, 1)
